In [1]:
!pip install faiss-gpu-cu12

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 33.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you 

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
import os
import cv2
import json
import glob
from tqdm.notebook import tqdm

In [3]:
!rsync -a --info=progress2 "/content/drive/MyDrive/Datasets/DINO_DatasetA.zip" "/content"
import zipfile
with zipfile.ZipFile("DINO_DatasetA.zip", 'r') as zip_ref:
    zip_ref.extractall("./dataset/")

     29,425,120 100%   29.41MB/s    0:00:00 (xfr#1, to-chk=0/1)


In [4]:
from transformers import AutoModelForImageClassification, AutoImageProcessor

#our ViT-S DINOv2, ImageNet weights are default
dinov2_vits14 = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14",pretrained=True)

device = torch.device('cuda' if torch.cuda.is_available() else "cpu")

processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small", use_fast=True)

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 261MB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

In [5]:
#assumes we're passed a train, val, or test set, and that all images are within class folders
def generate_dict_from_set(path_to_dataset, start_index):
  index = start_index
  all_embeddings = []
  for folder in os.listdir(path_to_dataset):
    label = folder
    fold_path = os.path.join(path_to_dataset, folder)
    for file in os.listdir(fold_path):
      filepath = os.path.join(fold_path, file)
      all_embeddings.append({'id': index, 'label':label, 'path': filepath})
      index += 1
  return all_embeddings

dataset = generate_dict_from_set("./dataset/train", 0)
print(dataset)

[{'id': 0, 'label': 'WS-P', 'path': './dataset/train/WS-P/GH016097_18.jpg'}, {'id': 1, 'label': 'WS-P', 'path': './dataset/train/WS-P/GH016076_2.jpg'}, {'id': 2, 'label': 'WS-P', 'path': './dataset/train/WS-P/GH016352_1.jpg'}, {'id': 3, 'label': 'WS-P', 'path': './dataset/train/WS-P/GH016057_0.jpg'}, {'id': 4, 'label': 'WS-P', 'path': './dataset/train/WS-P/GH016054_1.jpg'}, {'id': 5, 'label': 'WS-P', 'path': './dataset/train/WS-P/GH016075_1.jpg'}, {'id': 6, 'label': 'WS-P', 'path': './dataset/train/WS-P/GH016352_4.jpg'}, {'id': 7, 'label': 'WS-P', 'path': './dataset/train/WS-P/GH016348_2.jpg'}, {'id': 8, 'label': 'WS-P', 'path': './dataset/train/WS-P/GH016076_3.jpg'}, {'id': 9, 'label': 'WS-P', 'path': './dataset/train/WS-P/GH016075_5.jpg'}, {'id': 10, 'label': 'WS-P', 'path': './dataset/train/WS-P/GH016052_0.jpg'}, {'id': 11, 'label': 'WS-P', 'path': './dataset/train/WS-P/GH016353_5.jpg'}, {'id': 12, 'label': 'WS-P', 'path': './dataset/train/WS-P/GH016077_2.jpg'}, {'id': 13, 'label': 

In [6]:
def get_label_path_index(dataset_entry, index):
  id = dataset_entry['id']
  id = np.array([id], dtype='int64') #needs to be a 1d nparray for faiss
  label = dataset_entry['label']
  path = dataset_entry['path']
  return id, label, path

In [7]:


transform_image = T.Compose([T.ToTensor(),
                             T.Resize((224, 224)),
                             #T.CenterCrop(224), images are already cropped to the bird via yolo
                             T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]) #normalizing to imagenet's means/std

def load_image(img: str) -> torch.Tensor:
    """
    Load an image and return a tensor that can be used as an input to DINOv2.
    """
    img = Image.open(img)

    transformed_img = transform_image(img)[:3].unsqueeze(0)

    return transformed_img

In [8]:
import faiss
from faiss import normalize_L2
def create_index(files: list, output_path) -> faiss.IndexFlatIP:
    """
    Create an index that contains all of the images in the specified list of files.
    """
    #
    dimension = 384 #we already know DINOv2 outputs this dimension
    index = faiss.IndexFlatIP(dimension) #inner product index
    index = faiss.IndexIDMap(index)


    with torch.no_grad():
      for i, entry in enumerate(tqdm(files)):
        id, label, img_path = get_label_path_index(entry, i)

        embeddings = dinov2_vits14(load_image(img_path).to(device))

        embedding = embeddings[0].cpu().numpy()

        embedding = np.array(embedding).reshape(1, -1)

        #vectors need to be normalized both before adding to the index and before searching
        normalize_L2(embedding) #for the sake of using cosine similarity

        index.add_with_ids(embedding, id)

    with open(output_path + '.paths.json', 'w') as f:
      json.dump(files, f)

    faiss.write_index(index, OUTPUT_INDEX_PATH)
    print(f"Index created and saved to {output_path}")

    return index

OUTPUT_INDEX_PATH = "/content/vector.index"
faiss_index = create_index(dataset, OUTPUT_INDEX_PATH)

  0%|          | 0/807 [00:00<?, ?it/s]

Index created and saved to /content/vector.index


In [ ]:
!rsync -a --info=progress2 "/content/drive/MyDrive/FAISS_index/DINOv2/Dataset_B/vector.index.paths.json" "./"
!rsync -a --info=progress2 "/content/drive/MyDrive/FAISS_index/DINOv2/Dataset_B/vector.index" "./"

        185,667 100%  145.82MB/s    0:00:00 (xfr#1, to-chk=0/1)
      3,758,186 100%   38.20MB/s    0:00:00 (xfr#1, to-chk=0/1)


In [9]:
!cp vector.index.paths.json /content/drive/MyDrive/FAISS_index/DINOv2/Dataset_A/
!cp vector.index /content/drive/MyDrive/FAISS_index/DINOv2/Dataset_A/

In [10]:
import faiss
import json

#https://towardsdatascience.com/building-an-image-similarity-search-engine-with-faiss-and-clip-2211126d08fa/ again
def load_faiss_index(index_path):
    index = faiss.read_index(index_path)
    with open(index_path + '.paths.json', 'r') as f:
        image_paths = json.load(f)
    print(f"Index loaded from {index_path}")
    return index, image_paths

OUTPUT_INDEX_PATH = "/content/vector.index"
faiss_index, files = load_faiss_index(OUTPUT_INDEX_PATH)


Index loaded from /content/vector.index


In [11]:
from collections import Counter
import statistics
from faiss import normalize_L2

def majority_voting_cosine(faiss_index, embeddings, files):

  correct_guesses = []
  correct_distances = []
  incorrect_distances = []
  incorrect_guesses = []
  all_distances = []

  for i, entry in enumerate(tqdm(files)):
    id, label, img_path = get_label_path_index(entry, i)

    with torch.no_grad():
      query_vectors = dinov2_vits14(load_image(img_path).to(device))

      query_vector = query_vectors[0].cpu().numpy()

      query_vector = np.array(query_vector).reshape(1, -1)

      normalize_L2(query_vector) #have to normalize to do cosine similarity


    k = 5  # Number of nearest neighbors to retrieve
    distances, indices = faiss_index.search(query_vector, k)

    all_guesses = [] #all labels of nearest neighbors
    neighbor_distances = []

    for i, index in enumerate(indices[0]):
      #print(f"index: {index}")
      #print(f"files len: {len(files)}")
      distance = distances[0][i]

      #kind of a mess. getting the associated embeddings id/label with our neighbor index
      id = embeddings[index]['id']
      new_label = embeddings[index]['label']

      all_guesses.append(new_label)
      all_distances.append(distance)
      neighbor_distances.append(distance)
      print(f"Nearest neighbor {i+1}: {id}, {new_label} Distance {distance}")

    majority_vote = Counter(all_guesses)
    winner = sorted(all_guesses, key=lambda x: majority_vote[x], reverse=True)[0]
    if winner == label:
      print(f"most common label was {winner} which == original label {label}")
      correct_guesses.append(winner)
      correct_distances.extend(neighbor_distances)
    else:
      print(f"most common label was {winner} which != {label}")
      incorrect_guesses.append(winner)
      incorrect_distances.extend(neighbor_distances)

  print(f"Total accuracy: {len(correct_guesses)/(len(correct_guesses)+len(incorrect_guesses))}")

  print(f"Median of all distances: {statistics.median(all_distances)}")

  print(f"Median distance of incorrect guesses: {statistics.median(incorrect_distances)}")

  print(f"Median distance of correct guesses: {statistics.median(correct_distances)}")

  print(f"Lowest: {min(all_distances)} highest: {max(all_distances)}")



In [12]:
val_dataset = generate_dict_from_set("./dataset/val", 0)
majority_voting_cosine(faiss_index, files, val_dataset)

  0%|          | 0/176 [00:00<?, ?it/s]

Nearest neighbor 1: 28, WS-P Distance 0.9220223426818848
Nearest neighbor 2: 86, WS-P Distance 0.9194843173027039
Nearest neighbor 3: 57, WS-P Distance 0.9068142771720886
Nearest neighbor 4: 34, WS-P Distance 0.9038106203079224
Nearest neighbor 5: 75, WS-P Distance 0.8952693939208984
most common label was WS-P which == original label WS-P
Nearest neighbor 1: 69, WS-P Distance 0.9411085247993469
Nearest neighbor 2: 83, WS-P Distance 0.9376119375228882
Nearest neighbor 3: 18, WS-P Distance 0.9262661933898926
Nearest neighbor 4: 47, WS-P Distance 0.9260069727897644
Nearest neighbor 5: 68, WS-P Distance 0.9235782623291016
most common label was WS-P which == original label WS-P
Nearest neighbor 1: 94, WS-P Distance 0.9189790487289429
Nearest neighbor 2: 56, WS-P Distance 0.889631450176239
Nearest neighbor 3: 48, WS-P Distance 0.885570764541626
Nearest neighbor 4: 11, WS-P Distance 0.8818500638008118
Nearest neighbor 5: 46, WS-P Distance 0.8812106251716614
most common label was WS-P which ==

In [ ]:
def get_last_id(files):
  return files[-1]['id']

In [ ]:
!rsync -a --info=progress2 "drive/MyDrive/Datasets/masked-Dataset_C.zip" "./"
import zipfile
with zipfile.ZipFile("masked-Dataset_C.zip", 'r') as zip_ref:
    zip_ref.extractall("./dataset-C/")

    101,083,341 100%   95.70MB/s    0:00:01 (xfr#1, to-chk=0/1)


In [ ]:
id = get_last_id(files) + 1
val_dataset = generate_dict_from_set("./dataset-C/", id)
majority_voting_cosine(faiss_index, files, val_dataset)

  0%|          | 0/5218 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
Nearest neighbor 4: 249, K-WP Distance 0.3266324996948242
Nearest neighbor 5: 219, K-WP Distance 0.3274573087692261
most common label was K-WP which != RM-X
Nearest neighbor 1: 2420, L-MB Distance 0.2996430993080139
Nearest neighbor 2: 219, K-WP Distance 0.3333553969860077
Nearest neighbor 3: 1661, MO-R Distance 0.339150071144104
Nearest neighbor 4: 1592, MO-R Distance 0.33993667364120483
Nearest neighbor 5: 1588, MO-R Distance 0.3426821529865265
most common label was MO-R which != RM-X
Nearest neighbor 1: 1539, MO-R Distance 0.24571216106414795
Nearest neighbor 2: 1466, MO-R Distance 0.2665947675704956
Nearest neighbor 3: 1574, MO-R Distance 0.26817822456359863
Nearest neighbor 4: 1513, MO-R Distance 0.2695402204990387
Nearest neighbor 5: 1628, MO-R Distance 0.2713768482208252
most common label was MO-R which != RM-X
Nearest neighbor 1: 967, -Y Distance 0.3646027445793152
Nearest neighbor 2: 432, -Y Distance 0.3881685435771942
Nearest